# 04 — Plotting

Tour of the diagnostic plots in `scboost_ensemble.plots`. Uses the same tiny synthetic dataset as Notebook 01.

In [ ]:
import numpy as np, pandas as pd, anndata as ad
import matplotlib.pyplot as plt
from scboost_ensemble import StackingEnsembleDonorMetaCached, plots

In [ ]:
# Tiny synthetic dataset (same as Notebook 01)
rng = np.random.default_rng(42)
n_donors, n_types, n_per = 5, 4, 20; n = n_donors * n_types * n_per
donor_ids   = np.repeat(np.arange(n_donors), n_types * n_per)
cell_types  = np.tile(np.repeat(np.arange(n_types), n_per), n_donors)
cell_type_names = np.array(['T cell', 'B cell', 'Macrophage', 'Endothelial'])[cell_types]
donor_names = np.array([f'donor{i}' for i in donor_ids])
centers_A = rng.normal(0, 1, size=(n_types, 8))
centers_B = rng.normal(0, 1, size=(n_types, 8))
adata = ad.AnnData(
    X=rng.random((n, 10), dtype=np.float32),
    obs=pd.DataFrame({'cell_type': cell_type_names, 'batch': donor_names,
                       'donor_id': donor_names},
                      index=[f'cell_{i:04d}' for i in range(n)]),
    obsm={'embA': (centers_A[cell_types] + rng.normal(0, 0.3, (n,8))).astype(np.float32),
          'embB': (centers_B[cell_types] + rng.normal(0, 0.3, (n,8))).astype(np.float32)},
)
adata.obs['split'] = np.where(adata.obs['donor_id'] == 'donor4', 'test', 'train')
train = adata[adata.obs['split'] == 'train'].copy()
test  = adata[adata.obs['split'] == 'test'].copy()
stacker = StackingEnsembleDonorMetaCached(
    obsm_keys=['embA', 'embB'], cv_scheme='donor', n_folds=4,
    early_stopping_rounds=5, device='cpu', use_batch_feature=False)
stacker.fit(train)

## Method importance

In [ ]:
fig, ax = plt.subplots(figsize=(6,3))
plots.plot_method_importance(stacker, ax=ax); plt.show()

## Training summary (best iteration per fold)

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
plots.plot_training_summary(stacker, ax=ax); plt.show()

## OOF meta-features diagnostic

Projects the (n_train, n_methods × n_classes) OOF prediction matrix to 2D. If cell types cluster cleanly, base learners already do most of the work and the meta-learner is essentially a calibrator. Diffuse / overlapping clusters → harder meta job.

In [ ]:
fig, ax = plt.subplots(figsize=(6,5))
plots.plot_oof_meta_features(stacker, train, max_points=400, projection='pca', ax=ax)
plt.show()

## Per-cell-type accuracy from a classification report
Use after running `scboost fit --out_dir X/` — reads the saved TSV.

In [ ]:
# Example: write a synthetic classification report
from sklearn.metrics import classification_report
from scboost_ensemble.eval import build_classification_report
preds = stacker.predict(test)
proba = stacker.predict_proba(test)
report_df = build_classification_report(
    np.asarray(test.obs['cell_type']), preds, proba,
    stacker.label_encoder_, train)
fig, ax = plt.subplots(figsize=(7,4))
plots.plot_classification_report(report_df, top_n=10, sort='f1-score', ax=ax)
plt.show()